# Retinal Vessel Segmentation

**Group:**
* Jakub Biernat 160248
* Eryk Masian 160228

**Technologies Used:**
* **Language:** Python
* **Libraries:** TODO

## Imports

In [46]:
#General imports
from skimage import io
import ipywidgets as widgets
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import numpy as np
from sklearn.metrics import confusion_matrix, accuracy_score
from imblearn.metrics import geometric_mean_score, specificity_score, sensitivity_score

#Imports for Retinal Vessel Detection via Image Processing
from skimage.filters import threshold_otsu, gaussian, sobel
from skimage.morphology import opening, closing
from skimage.color import rgb2gray
from skimage import exposure

## Images
From HRF image database: https://www5.cs.fau.de/research/data/fundus-images/

In [47]:
image_names = [f"{str(i).zfill(2)}_{suffix}" for i in range(1, 16) for suffix in ["h", "g", "dr"]]

def load_images(image_name):
    raw_image = io.imread(f"../data/images/{image_name}.jpg")
    gs_image = io.imread(f"../data/goldstandard/{image_name}.tif")
    mask = io.imread(f"../data/fovs/{image_name}_mask.tif")
    mask = mask[..., 0]
    return raw_image, gs_image, mask


## Quality metrics and visualisation

In [48]:
def generate_gs_comparison(gs_image, generated_image):
    gs = gs_image > 0
    pred = generated_image > 0

    comparison_image = np.zeros((*generated_image.shape, 3), dtype=np.uint8)

    #True Positive
    tp = gs & pred
    comparison_image[tp] = [255, 255, 255]

    #False Positive
    fp = pred & ~gs
    comparison_image[fp] = [255, 0, 0]

    #False Negative
    fn = gs & ~pred
    comparison_image[fn] = [0, 0, 255]

    return comparison_image

def generate_overlay(raw_image, mask_image):
    pred = mask_image > 0

    overlay = raw_image.copy()
    overlay[pred] = [0, 255, 0]

    return overlay

def calculate_metrics(gs_image, generated_image, mask = None):
    valid = mask > 0

    y_true = (gs_image > 0)[valid].astype(int)
    y_generated = (generated_image > 0)[valid].astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_generated, labels=[0, 1]).ravel()

    accuracy = accuracy_score(y_true, y_generated)

    sensitivity = sensitivity_score(y_true, y_generated)

    specificity = specificity_score(y_true, y_generated)

    gmean = geometric_mean_score(y_true, y_generated, average='binary')

    return {
        "TP": tp,
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "Accuracy": accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "G-Mean": gmean
    }

## Retinal Vessel Detection via Image Processing

### Image processing

In [49]:
def retinal_vessel_segmentation_image_processing(image, mask):
    ######## Pre-processing ########
    image = rgb2gray(image)

    image = gaussian(image, sigma=1)

    image = exposure.equalize_hist(image)

    ######## Core processing ########
    image = sobel(image)

    ######## Post-processing ########
    image = image > threshold_otsu(image)

    image = closing(image)
    image = opening(image)

    return image

### App

In [50]:
# ---------------- UI ----------------
image_selector = widgets.Dropdown(
    options=image_names,
    description="Obraz:"
)

segment_button = widgets.Button(
    description="Segmentacja",
    button_style="success",
    icon="play"
)

preview_output = widgets.Output()
result_output = widgets.Output()
metrics_output = widgets.Output()
# ---------------- CLEANING ----------------
def on_image_change(change):
    with preview_output:
        clear_output(wait=True)
    with result_output:
        clear_output(wait=True)
    with metrics_output:
        clear_output(wait=True)

    show_preview()

# ---------------- PREVIEW ----------------
def show_preview(change=None):

    with preview_output:
        clear_output(wait=True)

        image_name = image_selector.value
        raw_image, _, _ = load_images(image_name)

        plt.figure(figsize=(5, 5))
        plt.imshow(raw_image, cmap="gray")
        plt.title(image_name)
        plt.axis("off")
        plt.show()

# ---------------- SEGMENTATION ----------------
def run_segmentation(button):

    with result_output:
        clear_output(wait=True)

        image_name = image_selector.value

        raw_image, gs_image, mask = load_images(image_name)

        generated_image = retinal_vessel_segmentation_image_processing(raw_image, mask)
        comparison_image = generate_gs_comparison(gs_image, generated_image)
        overlay = generate_overlay(raw_image, generated_image)
        gs_overlay = generate_overlay(raw_image, gs_image)


        fig, axes = plt.subplots(2, 3, figsize=(15, 8))

        axes[0][0].imshow(generated_image, cmap="gray")
        axes[0][0].set_title("Wygenerowana segmentacja")
        axes[0][0].axis("off")

        axes[0][1].imshow(gs_image, cmap="gray")
        axes[0][1].set_title("Maska ekspercka")
        axes[0][1].axis("off")

        axes[0][2].imshow(comparison_image, cmap="gray")
        axes[0][2].set_title("Porównanie wygenerowanej segmentacji\ni maski eksperckiej")
        axes[0][2].axis("off")

        axes[1][0].imshow(raw_image)
        axes[1][0].set_title("Obraz wejściowy")
        axes[1][0].axis("off")

        axes[1][1].imshow(overlay)
        axes[1][1].set_title("Wykryte naczynia na obrazie wejściowym")
        axes[1][1].axis("off")

        axes[1][2].imshow(gs_overlay)
        axes[1][2].set_title("Maska ekspercka na obrazie wejściowym")
        axes[1][2].axis("off")

        plt.tight_layout()
        plt.show()

    metrics = calculate_metrics(gs_image, generated_image, mask)
    with metrics_output:
        clear_output(wait=True)

        print("Metryki:")

        for k, v in metrics.items():
            if isinstance(v, float):
                print(f"{k:12s}: {v:.4f}")
            else:
                print(f"{k:12s}: {v}")

# ---------------- OBSERVERS ----------------
image_selector.observe(on_image_change, names="value")
segment_button.on_click(run_segmentation)

# ---------------- LAYOUT ----------------
left_panel = widgets.VBox([
    image_selector,
    segment_button
])

top_panel = widgets.HBox([
    left_panel,
    preview_output
])

display(
    widgets.VBox([
        top_panel,
        result_output,
        metrics_output
    ])
)

# pierwszy podgląd
show_preview()